# 01 — Exploration & Analyse descriptive
## RobynMMM · Business Scientist MCE · Ekimetrics

**Objectif :** comprendre la structure des données, les distributions, les corrélations et les patterns temporels avant toute modélisation.

**Dataset :** Robyn (Meta) — marque FMCG anonymisée · 208 semaines · 5 canaux media

---
### Plan
1. Chargement & validation des données
2. Statistiques descriptives
3. Analyse temporelle des ventes
4. Analyse des dépenses media
5. Corrélations & relations ventes/media
6. Saisonnalité & facteurs externes
7. Synthèse & insights clés

## 0. Imports & configuration

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
from pathlib import Path
import warnings
warnings.filterwarnings("ignore")

plt.rcParams.update({
    "figure.dpi": 120,
    "figure.facecolor": "white",
    "axes.spines.top": False,
    "axes.spines.right": False,
    "axes.grid": True,
    "grid.alpha": 0.3,
    "font.size": 11,
})

MEDIA_COLS = ["tv_spend", "ooh_spend", "print_spend", "facebook_spend", "search_spend"]
MEDIA_LABELS = {
    "tv_spend": "TV", "ooh_spend": "OOH", "print_spend": "Print",
    "facebook_spend": "Facebook", "search_spend": "Search"
}
MEDIA_COLORS = {
    "tv_spend": "#4E9AF1", "ooh_spend": "#F5C842",
    "print_spend": "#6BCB77", "facebook_spend": "#FF7043", "search_spend": "#9C5CF5"
}
Path("../outputs/figures").mkdir(parents=True, exist_ok=True)
print("[OK] Imports charges")

## 1. Chargement & validation des données

Le dataset Robyn est disponible sur le repo officiel Meta.
**Si le téléchargement échoue**, place `dt_simulated_weekly.csv` dans `data/raw/` manuellement.

In [ ]:
import urllib.request

DATA_DIR = Path("../data/raw")
DATA_DIR.mkdir(parents=True, exist_ok=True)
DATA_PATH = DATA_DIR / "dt_simulated_weekly.csv"

if not DATA_PATH.exists():
    print("Telechargement du dataset Robyn...")
    url = "https://raw.githubusercontent.com/facebookexperimental/Robyn/main/robyn/data/dt_simulated_weekly.csv"
    try:
        req = urllib.request.Request(url, headers={"User-Agent": "Mozilla/5.0"})
        with urllib.request.urlopen(req, timeout=30) as r:
            DATA_PATH.write_bytes(r.read())
        print(f"[OK] Telecharge -> {DATA_PATH}")
    except Exception as e:
        print(f"[WARN] Echec telechargement : {e}")
        print("-> Place manuellement dt_simulated_weekly.csv dans data/raw/")
else:
    print(f"[OK] Fichier present : {DATA_PATH}")

df = pd.read_csv(DATA_PATH, parse_dates=["date"])
df = df.sort_values("date").reset_index(drop=True)

print(f"Shape        : {df.shape[0]} lignes x {df.shape[1]} colonnes")
print(f"Periode      : {df['date'].min().date()} -- {df['date'].max().date()}")
print(f"Frequence    : hebdomadaire")
print(f"\nColonnes :")
print(df.dtypes.to_string())

### Validation qualité

In [ ]:
missing = df.isnull().sum()
print("=== Valeurs manquantes ===")
if missing.sum() == 0:
    print("Aucune valeur manquante [OK]")
else:
    print(missing[missing > 0])

dupes = df.duplicated(subset=["date"]).sum()
print(f"\nDoublons sur date : {dupes}")

print("\n=== Apercu (5 premieres lignes) ===")
df.head()

## 2. Statistiques descriptives

In [ ]:
media_present = [c for c in MEDIA_COLS if c in df.columns]
cols_desc = ["revenue"] + media_present

stats = df[cols_desc].describe().T
stats["cv_%"] = (stats["std"] / stats["mean"] * 100).round(1)
stats["total_M"] = (df[cols_desc].sum() / 1e6).round(2)

print("=== Statistiques descriptives ===")
display(stats[["mean", "std", "cv_%", "min", "max", "total_M"]].round(0))

In [ ]:
total_media = df[media_present].sum().sum()
print("=== Share of Voice (% budget media total) ===")
for col in media_present:
    pct = df[col].sum() / total_media * 100
    bar = chr(9608) * int(pct/2) + chr(9617) * (50 - int(pct/2))
    print(f"  {MEDIA_LABELS[col]:<12} {pct:>5.1f}%  {bar}")

print(f"\nBudget media total  : {total_media/1e6:.1f}M EUR")
print(f"Revenue total       : {df['revenue'].sum()/1e6:.1f}M EUR")
print(f"ROAS brut global    : {df['revenue'].sum() / total_media:.2f}x")

## 3. Analyse temporelle des ventes

In [ ]:
fig, axes = plt.subplots(3, 1, figsize=(14, 12))
fig.suptitle("Analyse temporelle des ventes", fontsize=14, fontweight="bold", y=1.01)

# Revenue serie temporelle
ax = axes[0]
ax.plot(df["date"], df["revenue"]/1000, color="#4E9AF1", linewidth=1.5, alpha=0.9, label="Revenue")
ax.fill_between(df["date"], df["revenue"]/1000, alpha=0.1, color="#4E9AF1")
ma8 = df["revenue"].rolling(8).mean()
ax.plot(df["date"], ma8/1000, color="#FF7043", linewidth=2, linestyle="--", label="MA 8 sem.")
if "promotion" in df.columns:
    for d in df[df["promotion"]==1]["date"]:
        ax.axvline(x=d, color="#6BCB77", alpha=0.25, linewidth=0.8)
ax.set_title("Revenue hebdomadaire (EUR K)")
ax.set_ylabel("Revenue (EUR K)")
ax.legend()
ax.annotate("lignes vertes = semaines promo", xy=(0.01, 0.05),
            xycoords="axes fraction", fontsize=9, color="#3a6a3a")

# Boxplot par annee
ax = axes[1]
df["year"] = df["date"].dt.year
yearly_data = [df[df["year"]==y]["revenue"].values/1000 for y in sorted(df["year"].unique())]
bp = ax.boxplot(yearly_data, patch_artist=True,
                labels=[str(y) for y in sorted(df["year"].unique())],
                medianprops={"color": "#FF7043", "linewidth": 2})
for patch in bp["boxes"]:
    patch.set_facecolor("#4E9AF14D")
ax.set_title("Distribution du revenue par annee")
ax.set_ylabel("Revenue (EUR K)")

# Saisonnalite mensuelle
ax = axes[2]
df["month"] = df["date"].dt.month
monthly_avg = df.groupby("month")["revenue"].mean() / 1000
monthly_std = df.groupby("month")["revenue"].std() / 1000
months_labels = ["Jan","Fev","Mar","Avr","Mai","Jun","Jul","Aou","Sep","Oct","Nov","Dec"]
bars = ax.bar(range(1, 13), monthly_avg, color="#4E9AF1", alpha=0.8)
ax.errorbar(range(1, 13), monthly_avg, yerr=monthly_std,
            fmt="none", color="gray", capsize=4, linewidth=1.5)
ax.set_xticks(range(1, 13))
ax.set_xticklabels(months_labels)
ax.set_title("Revenue moyen par mois (saisonnalite)")
ax.set_ylabel("Revenue moyen (EUR K)")
bars[11].set_color("#F5C842")
bars[11].set_alpha(1.0)

plt.tight_layout()
plt.savefig("../outputs/figures/01_revenue_temporel.png", bbox_inches="tight", dpi=150)
plt.show()
print("[OK] Figure sauvegardee")

## 4. Analyse des dépenses media

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
fig.suptitle("Analyse des depenses media", fontsize=14, fontweight="bold")

# Stacked area
ax = axes[0, 0]
bottom = np.zeros(len(df))
for col in media_present:
    vals = df[col].values / 1000
    ax.fill_between(df["date"], bottom, bottom + vals,
                   label=MEDIA_LABELS[col], alpha=0.85, color=MEDIA_COLORS[col])
    bottom += vals
ax.set_title("Depenses media hebdomadaires (stacked)")
ax.set_ylabel("Depenses (EUR K)")
ax.legend(loc="upper left", fontsize=9)

# Donut SOV
ax = axes[0, 1]
sizes = [df[c].sum() for c in media_present]
labels = [MEDIA_LABELS[c] for c in media_present]
colors = [MEDIA_COLORS[c] for c in media_present]
wedges, texts, autotexts = ax.pie(
    sizes, labels=labels, colors=colors, autopct="%1.1f%%",
    pctdistance=0.8, startangle=90, wedgeprops={"width": 0.5})
for at in autotexts: at.set_fontsize(9)
ax.set_title("Share of Voice (budget total)")

# Violin
ax = axes[1, 0]
data_v = [df[c].values/1000 for c in media_present]
parts = ax.violinplot(data_v, positions=range(len(media_present)), showmeans=True, showmedians=True)
for pc, col in zip(parts["bodies"], [MEDIA_COLORS[c] for c in media_present]):
    pc.set_facecolor(col); pc.set_alpha(0.7)
ax.set_xticks(range(len(media_present)))
ax.set_xticklabels([MEDIA_LABELS[c] for c in media_present], fontsize=9)
ax.set_title("Distribution des depenses hebdomadaires")
ax.set_ylabel("Depenses (EUR K)")

# Boxplot
ax = axes[1, 1]
bp = ax.boxplot(data_v, patch_artist=True,
                labels=[MEDIA_LABELS[c] for c in media_present],
                medianprops={"color": "black", "linewidth": 2},
                flierprops={"marker": "o", "markersize": 4, "alpha": 0.5})
for patch, col in zip(bp["boxes"], [MEDIA_COLORS[c] for c in media_present]):
    patch.set_facecolor(col); patch.set_alpha(0.7)
ax.set_title("Boxplot des depenses par canal")
ax.set_ylabel("Depenses (EUR K)")

plt.tight_layout()
plt.savefig("../outputs/figures/01_media_analysis.png", bbox_inches="tight", dpi=150)
plt.show()
print("[OK] Figure sauvegardee")

## 5. Corrélations & relations ventes / media

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 6))
fig.suptitle("Correlations ventes / media", fontsize=14, fontweight="bold")

cols_corr = ["revenue"] + media_present + (["price_index"] if "price_index" in df.columns else []) + (["promotion"] if "promotion" in df.columns else [])
corr = df[cols_corr].corr()
mask = np.triu(np.ones_like(corr, dtype=bool))

ax = axes[0]
labels_short = [c.replace("_spend","").replace("_index","") for c in cols_corr]
sns.heatmap(corr, mask=mask, ax=ax, annot=True, fmt=".2f",
            cmap="RdYlGn", center=0, vmin=-1, vmax=1,
            square=True, linewidths=0.5, cbar_kws={"shrink": 0.8},
            xticklabels=labels_short, yticklabels=labels_short)
ax.set_title("Matrice de correlation")
ax.tick_params(axis="x", rotation=45)

ax = axes[1]
corr_rev = corr["revenue"].drop("revenue").sort_values(ascending=True)
colors_bar = ["#FF7043" if v < 0 else "#4E9AF1" for v in corr_rev.values]
labels_bar = [c.replace("_spend","").replace("_index","") for c in corr_rev.index]
bars = ax.barh(labels_bar, corr_rev.values, color=colors_bar, alpha=0.8)
ax.axvline(x=0, color="black", linewidth=0.8)
ax.set_title("Correlation avec le revenue")
ax.set_xlabel("Correlation de Pearson")
for bar, val in zip(bars, corr_rev.values):
    ax.text(val + 0.005 * np.sign(val), bar.get_y() + bar.get_height()/2,
            f"{val:.2f}", va="center", fontsize=10, fontweight="bold")

plt.tight_layout()
plt.savefig("../outputs/figures/01_correlations.png", bbox_inches="tight", dpi=150)
plt.show()

print("\n=== Correlations avec revenue ===")
for c, v in corr_rev.sort_values(ascending=False).items():
    label = c.replace("_spend","").replace("_index","")
    tag = "[fort]" if abs(v)>0.3 else "[modere]" if abs(v)>0.15 else "[faible]"
    print(f"  {label:<15} r={v:+.3f}  {tag}")

### Scatter plots : dépenses vs revenue

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(15, 9))
fig.suptitle("Relation depenses -> revenue (scatter plots)", fontsize=14, fontweight="bold")
axes_flat = axes.flatten()

for i, col in enumerate(media_present):
    ax = axes_flat[i]
    color = MEDIA_COLORS[col]
    ax.scatter(df[col]/1000, df["revenue"]/1000,
               alpha=0.5, s=25, color=color, edgecolors="white", linewidth=0.3)
    z = np.polyfit(df[col], df["revenue"], 1)
    p = np.poly1d(z)
    x_line = np.linspace(df[col].min(), df[col].max(), 100)
    ax.plot(x_line/1000, p(x_line)/1000, color="black", linewidth=1.5, linestyle="--", alpha=0.7)
    r = df[col].corr(df["revenue"])
    ax.set_title(f"{MEDIA_LABELS[col]}  (r={r:.2f})", fontweight="bold")
    ax.set_xlabel(f"{MEDIA_LABELS[col]} spend (EUR K)")
    ax.set_ylabel("Revenue (EUR K)")
    if "promotion" in df.columns:
        mask = df["promotion"] == 1
        ax.scatter(df[mask][col]/1000, df[mask]["revenue"]/1000,
                   alpha=0.8, s=40, color="#FF7043", zorder=5, label="Promo")
        if i == 0: ax.legend(fontsize=8)

for j in range(len(media_present), 6):
    axes_flat[j].set_visible(False)

plt.tight_layout()
plt.savefig("../outputs/figures/01_scatter_media_revenue.png", bbox_inches="tight", dpi=150)
plt.show()
print("[OK] Figure sauvegardee")

## 6. Saisonnalité & facteurs externes

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
fig.suptitle("Facteurs externes et saisonnalite", fontsize=14, fontweight="bold")

# Promo vs non-promo
ax = axes[0, 0]
if "promotion" in df.columns:
    data_promo = [df[df["promotion"]==0]["revenue"]/1000,
                  df[df["promotion"]==1]["revenue"]/1000]
    bp = ax.boxplot(data_promo, patch_artist=True,
                    labels=["Sans promo","Avec promo"],
                    medianprops={"color":"black","linewidth":2})
    bp["boxes"][0].set_facecolor("#4E9AF14D")
    bp["boxes"][1].set_facecolor("#6BCB774D")
    lift = (df[df["promotion"]==1]["revenue"].mean() /
            df[df["promotion"]==0]["revenue"].mean() - 1) * 100
    ax.set_title(f"Impact des promotions (lift : +{lift:.1f}%)")
    ax.set_ylabel("Revenue (EUR K)")

# Saisonnalite trimestrielle
ax = axes[0, 1]
df["quarter"] = df["date"].dt.quarter
q_avg = df.groupby(["year","quarter"])["revenue"].mean().reset_index()
for y in sorted(df["year"].unique()):
    sub = q_avg[q_avg["year"]==y]
    ax.plot(sub["quarter"], sub["revenue"]/1000, marker="o", label=str(y), linewidth=2, markersize=6)
ax.set_xticks([1,2,3,4])
ax.set_xticklabels(["T1","T2","T3","T4"])
ax.set_title("Revenue moyen par trimestre")
ax.set_ylabel("Revenue (EUR K)")
ax.legend(fontsize=9)

# Tendance (rolling mean)
ax = axes[1, 0]
ax.plot(df["date"], df["revenue"]/1000, alpha=0.4, color="#4E9AF1", label="Brut")
for w, c, lw in [(4,"#F5C842",1.5),(13,"#FF7043",2),(26,"#6BCB77",2.5)]:
    ax.plot(df["date"], df["revenue"].rolling(w).mean()/1000,
            color=c, linewidth=lw, label=f"MA {w} sem.")
ax.set_title("Decomposition tendance (moyennes mobiles)")
ax.set_ylabel("Revenue (EUR K)")
ax.legend(fontsize=9)

# Prix vs revenue
ax = axes[1, 1]
if "price_index" in df.columns:
    ax2 = ax.twinx()
    ax.plot(df["date"], df["price_index"], color="#9C5CF5", linewidth=1.5, alpha=0.8, label="Prix")
    ax2.plot(df["date"], df["revenue"]/1000, color="#4E9AF1", linewidth=1, alpha=0.6, label="Revenue")
    ax.set_ylabel("Indice de prix", color="#9C5CF5")
    ax2.set_ylabel("Revenue (EUR K)", color="#4E9AF1")
    ax.set_title("Prix vs Revenue")
    r = df["price_index"].corr(df["revenue"])
    ax.annotate(f"r = {r:.2f}", xy=(0.05, 0.92), xycoords="axes fraction",
                fontsize=10, color="#9C5CF5", fontweight="bold")

plt.tight_layout()
plt.savefig("../outputs/figures/01_saisonnalite.png", bbox_inches="tight", dpi=150)
plt.show()
print("[OK] Figure sauvegardee")

## 7. Synthèse & insights clés

In [ ]:
print("=" * 65)
print("  SYNTHESE EDA -- RobynMMM")
print("=" * 65)

rev_mean = df["revenue"].mean()/1000
rev_std  = df["revenue"].std()/1000
rev_cv   = df["revenue"].std()/df["revenue"].mean()*100
budget   = df[media_present].sum().sum()/1e6
tv_pct   = df["tv_spend"].sum()/df[media_present].sum().sum()*100 if "tv_spend" in df.columns else 0
fb_pct   = df["facebook_spend"].sum()/df[media_present].sum().sum()*100 if "facebook_spend" in df.columns else 0

print(f"\nDONNEES")
print(f"  Periode     : {df['date'].min().date()} -- {df['date'].max().date()}")
print(f"  Semaines    : {len(df)}")
print(f"  Revenue moy : {rev_mean:.0f}K EUR/sem  (sigma={rev_std:.0f}K)")
print(f"  Volatilite  : {rev_cv:.1f}% (CV)")
print(f"\nBUDGET MEDIA")
print(f"  Total       : {budget:.1f}M EUR sur la periode")
if tv_pct > 0: print(f"  TV          : {tv_pct:.0f}% du budget")
if fb_pct > 0: print(f"  Facebook    : {fb_pct:.0f}% du budget")
print(f"\nCORRELATIONS BRUTES (avant adstock/saturation)")
for c in media_present:
    r = df[c].corr(df["revenue"])
    tag = "[fort]" if abs(r)>0.3 else "[modere]" if abs(r)>0.15 else "[faible]"
    print(f"  {MEDIA_LABELS[c]:<12} r={r:+.3f}  {tag}")

if "promotion" in df.columns:
    lift = (df[df["promotion"]==1]["revenue"].mean() /
            df[df["promotion"]==0]["revenue"].mean() - 1) * 100
    print(f"\nFACTEURS EXTERNES")
    print(f"  Promotions : +{lift:.1f}% de revenue en semaines promo")

print("\nPROCHAINE ETAPE -- Notebook 02 : Feature engineering")
print("  - Appliquer adstock (Koyck decay) par canal")
print("  - Appliquer saturation (Hill function) par canal")
print("  - Ces transformations augmenteront les correlations")
print("=" * 65)